# 作者级粉丝门槛探索（2026-10-08）

本笔记仅复算工作台作者栏的聚合筛选计数，不包含作者标识、原始UGC、凭据或源库。
小红书5,000和知乎10,000作为已有参照；以两平台作者数量加权其达到门槛比例，
为另外三平台寻找相近的上层比例。不得将本方法解释为识别出真实KOL边界。

四舍五入到百位是本次计算假设。各平台账号尚未区分官方与个人，因此结果不是
个人账号最终门槛；正式执行前应先检查个人子集。平台账号数不是跨平台去重人数。

该数据来自当日可见网页聚合，不是最新数据库文件的哈希冻结快照。运行下方两个
代码单元可复算全部输出，无需联网。正式论文数字应由仓库脚本对冻结资料重建。
讨论决定及下一步只记录在总备忘录B02/B06，不在本笔记维护待办。


In [1]:
# 页面聚合观测；嵌入去敏计数，以便离线复算。
observations = {'checked_on_local_date': '2026-10-08', 'timezone': 'Asia/Shanghai', 'source': '研究资料库网页作者栏', 'source_kind': '研究资料库作者栏筛选后显示的聚合计数；不是直接SQL导出', 'retained_posts': 10914, 'platform_accounts': 7886, 'count_fields': ['platform', 'minimum_raw_followers_inclusive', 'author_count'], 'counts': [['小红书', 4950, 995], ['小红书', 9950, 714], ['小红书', 950, 1799], ['知乎', 9950, 34], ['知乎', 4950, 40], ['知乎', 950, 98], ['抖音', 950, 330], ['抖音', 4950, 117], ['抖音', 9950, 79], ['抖音', 2450, 196], ['微博', 950, 836], ['微博', 4950, 526], ['微博', 9950, 468], ['微博', 4750, 530], ['B站', 950, 76], ['B站', 4950, 33], ['B站', 9950, 25], ['B站', 650, 86], ['B站', 550, 89], ['B站', 0, 391], ['B站', 450, 94], ['抖音', 2550, 191], ['抖音', 0, 825], ['微博', 4550, 536], ['微博', 4450, 540], ['微博', 4650, 531], ['微博', 0, 2281], ['小红书', 0, 4014], ['知乎', 0, 375]], 'exact_zero_counts': {'B站': 31}, 'population': '全部研究库平台账号，官方与个人尚未分离；同一平台账号按作者栏计数，未跨平台合并自然人', 'rounding_assumption': '非负整数四舍五入至百位；取整值达到T等价于原值至少T-50；作为探索假设，尚未冻结正式比较细则', 'provenance_limits': ['通过网页筛选聚合读取，未下载源库，因此没有本次源库哈希', '原始非负粉丝字段有值不等于来源或快照已验证', '无法从聚合计数计算完整均值、标准差或判定自然分界', '仅用于候选比较，不赋予账号标签，不以内容差异优化门槛'], 'workspace_head': 'origin/main@083cc22'}


In [2]:
"""复算作者级粉丝覆盖与候选门槛，不连接或修改源数据库。

输入为工作台可见聚合计数；官方账号尚未剥离。输出只描述当前样本的规模
位置，不声称粉丝数能够验证意见影响力。算法完全确定，不涉及随机抽样。
"""
from fractions import Fraction
import json

counts = {(p, low): n for p, low, n in observations["counts"]}
platforms = ["小红书", "知乎", "抖音", "微博", "B站"]
totals = {p: counts[p, 0] for p in platforms}
assert sum(totals.values()) == observations["platform_accounts"]
# 检查筛选结果的单调性，防止查询尚未更新或转录错误形成自相矛盾的计数。
for p in platforms:
    ordered = sorted((low, n) for (name, low), n in counts.items() if name == p)
    assert all(a[1] >= b[1] for a, b in zip(ordered, ordered[1:]))

anchors = {"小红书": 5000, "知乎": 10000}
anchor_n = sum(totals[p] for p in anchors)
target = Fraction(sum(counts[p, t - 50] for p, t in anchors.items()), anchor_n)
equal_platform_target = sum(Fraction(counts[p, t - 50], totals[p]) for p, t in anchors.items()) / len(anchors)
# 只在已查询的候选档位中选择覆盖率最接近参照者；这不是身份真值优化。
candidate_grids = {"抖音": [2500, 2600], "微博": [4500, 4600, 4700, 4800, 5000], "B站": [500, 600, 700, 1000]}
selected = dict(anchors)
for p, grid in candidate_grids.items():
    selected[p] = min(grid, key=lambda t: (abs(Fraction(counts[p, t - 50], totals[p]) - target), t))
summary = {
    "reference_author_count": anchor_n,
    "reference_above_anchor_count": sum(counts[p, t - 50] for p, t in anchors.items()),
    "reference_upper_share": float(target),
    "equal_platform_reference_upper_share": float(equal_platform_target),
    "anchor_weights": {p: totals[p] / anchor_n for p in anchors},
    "platforms": []
}
for p in platforms:
    threshold = selected[p]
    n_above = counts[p, threshold - 50]
    summary["platforms"].append({
        "platform": p, "authors": totals[p], "sample_weight": totals[p] / sum(totals.values()),
        "above_1000_count": counts[p, 950], "above_5000_count": counts[p, 4950], "above_10000_count": counts[p, 9950],
        "threshold": threshold, "above_threshold_count": n_above, "above_threshold_share": n_above / totals[p],
        "status": "existing_anchor" if p in anchors else "exploratory_candidate_all_accounts",
    })
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "reference_author_count": 4389,
  "reference_above_anchor_count": 1029,
  "reference_upper_share": 0.23444976076555024,
  "equal_platform_reference_upper_share": 0.16927453911310414,
  "anchor_weights": {
    "小红书": 0.9145591250854409,
    "知乎": 0.08544087491455912
  },
  "platforms": [
    {
      "platform": "小红书",
      "authors": 4014,
      "sample_weight": 0.5090032969819934,
      "above_1000_count": 1799,
      "above_5000_count": 995,
      "above_10000_count": 714,
      "threshold": 5000,
      "above_threshold_count": 995,
      "above_threshold_share": 0.2478824115595416,
      "status": "existing_anchor"
    },
    {
      "platform": "知乎",
      "authors": 375,
      "sample_weight": 0.04755262490489475,
      "above_1000_count": 98,
      "above_5000_count": 40,
      "above_10000_count": 34,
      "threshold": 10000,
      "above_threshold_count": 34,
      "above_threshold_share": 0.09066666666666667,
      "status": "existing_anchor"
    },
    {
      "platform"